
# `cylinder_nektar_wake.mat` 数据集理解与全场可视化

这个 notebook 的目的不是训练模型，而是把当前 Navier–Stokes benchmark **看明白**：

- `.mat` 里到底有哪些 key；
- `X_star / t / U_star / p_star` 各是什么维度；
- 空间是 2D 还是 3D；
- 每个 `(x,y,t)` 对应哪些物理量；
- 完整 reference field 长什么样；
- 当前实验随机抽取 800 个时空监督点到底有多稀疏；
- 如何画二维完整场、速度矢量场、三维场值图和 `x-y-t` 时空图；
- 可选：加载一个已经训练好的 `model.pt`，把 **reference / prediction / absolute error** 连起来理解。

## 先记住一个最重要的概念

这个数据集的物理空间是：

$$
(x,y),
$$

再加时间：

$$
t.
$$

所以它是 **2D space + time**，不是三维物理空间 CFD。

如果画三维图：

- `(x,y,t)`：第三轴是时间；
- `(x,y,u)` / `(x,y,v)` / `(x,y,p)`：第三轴是场值；

二者都不是物理空间中的 `z`。


In [ ]:

from pathlib import Path
import os
import sys
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

print("Python:", sys.version.split()[0])
print("NumPy :", np.__version__)


## 1. 定位并读取 `.mat` 文件

In [ ]:

# 默认优先使用你当前服务器上的 PINNsFormer 路径。
DATA_CANDIDATES = [
    Path(os.environ.get(
        "CYLINDER_WAKE_MAT",
        "/home/simplexity/cyt/pinnsformer-main/demo/navier_stokes/cylinder_nektar_wake.mat"
    )),
    Path("./cylinder_nektar_wake.mat"),
    Path("./demo/navier_stokes/cylinder_nektar_wake.mat"),
]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "没有找到 cylinder_nektar_wake.mat。\n"
        "请把环境变量 CYLINDER_WAKE_MAT 指向文件，"
        "或把文件放到当前目录。\n"
        "尝试过：\n" + "\n".join(str(p) for p in DATA_CANDIDATES)
    )

print("Using:", DATA_PATH.resolve())

data = scipy.io.loadmat(DATA_PATH)
keys = [k for k in data.keys() if not k.startswith("__")]
print("MAT keys:", keys)


## 2. 核心数组、维度与数据量

In [ ]:

required = ["X_star", "t", "U_star", "p_star"]
missing = [k for k in required if k not in data]
assert not missing, f"缺少预期变量: {missing}"

X_star = np.asarray(data["X_star"])
t_star = np.asarray(data["t"])
U_star = np.asarray(data["U_star"])
P_star = np.asarray(data["p_star"])

N = X_star.shape[0]
T = t_star.shape[0]

print("X_star:", X_star.shape, X_star.dtype, " -> spatial coordinates (x,y)")
print("t     :", t_star.shape, t_star.dtype, " -> time")
print("U_star:", U_star.shape, U_star.dtype, " -> velocity (u,v)")
print("p_star:", P_star.shape, P_star.dtype, " -> pressure")
print()
print("N spatial points :", N)
print("T time snapshots :", T)
print("N*T space-time points:", N * T)

assert X_star.ndim == 2 and X_star.shape[1] == 2
assert U_star.shape[0] == N and U_star.shape[1] == 2 and U_star.shape[2] == T
assert P_star.shape == (N, T)


In [ ]:

x_space = X_star[:, 0]
y_space = X_star[:, 1]
t_values = t_star.reshape(-1)

u_all = U_star[:, 0, :]  # N x T
v_all = U_star[:, 1, :]  # N x T
p_all = P_star            # N x T
speed_all = np.sqrt(u_all**2 + v_all**2)

print("Unique x coordinates:", len(np.unique(x_space)))
print("Unique y coordinates:", len(np.unique(y_space)))

if len(t_values) > 1:
    dt = np.diff(t_values)
    print("time range:", (float(t_values.min()), float(t_values.max())))
    print("dt min/max/mean:", float(dt.min()), float(dt.max()), float(dt.mean()))
else:
    print("Only one time point found.")


## 3. 数值范围：坐标、速度、压力分别有多大

In [ ]:

def describe(name, arr):
    arr = np.asarray(arr)
    finite = np.isfinite(arr)
    print(
        f"{name:10s} "
        f"shape={str(arr.shape):16s} "
        f"dtype={str(arr.dtype):10s} "
        f"finite={finite.mean()*100:7.3f}% "
        f"min={np.nanmin(arr): .6g} "
        f"max={np.nanmax(arr): .6g} "
        f"mean={np.nanmean(arr): .6g} "
        f"std={np.nanstd(arr): .6g}"
    )

describe("x", x_space)
describe("y", y_space)
describe("t", t_values)
describe("u", u_all)
describe("v", v_all)
describe("p", p_all)
describe("|vel|", speed_all)


## 4. 把 `.mat` 想象成普通表格

In [ ]:

# 与原始 PINN / 当前训练代码相同的展开方式。
XX = np.tile(X_star[:, 0:1], (1, T))
YY = np.tile(X_star[:, 1:2], (1, T))
TT = np.tile(t_star, (1, N)).T

x_flat = XX.reshape(-1)
y_flat = YY.reshape(-1)
t_flat = TT.reshape(-1)
u_flat = u_all.reshape(-1)
v_flat = v_all.reshape(-1)
p_flat = p_all.reshape(-1)

headers = ["x", "y", "t", "u", "v", "p"]
sample_idx = np.linspace(0, N*T - 1, num=min(10, N*T), dtype=int)
sample_rows = np.column_stack([
    x_flat[sample_idx],
    y_flat[sample_idx],
    t_flat[sample_idx],
    u_flat[sample_idx],
    v_flat[sample_idx],
    p_flat[sample_idx],
])

print(" | ".join(f"{h:>12s}" for h in headers))
print("-" * 91)
for row in sample_rows:
    print(" | ".join(f"{float(v):12.6f}" for v in row))



上面的每一行都可以理解为：

$$
(x_i,y_i,t_j)\rightarrow(u_{ij},v_{ij},p_{ij}).
$$

`U_star` / `p_star` 是完整 reference field；当前 PINNsFormer + ConFIG 训练并不会把这 100 万个位置全部作为监督样本。


## 5. 空间采样区域：5000 个点到底分布在哪里

In [ ]:

plt.figure(figsize=(8, 4.5))
plt.scatter(x_space, y_space, s=6)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Spatial coordinates X_star | N={N}")
plt.axis("equal")
plt.tight_layout()
plt.show()


## 6. 固定一个时间快照，直接看完整 reference field

In [ ]:

SNAPSHOT = min(100, T - 1)

u_snap = u_all[:, SNAPSHOT]
v_snap = v_all[:, SNAPSHOT]
p_snap = p_all[:, SNAPSHOT]
speed_snap = speed_all[:, SNAPSHOT]
t_snap = float(t_values[SNAPSHOT])

print("SNAPSHOT index:", SNAPSHOT)
print("physical time :", t_snap)


In [ ]:

def plot_2d_field(values, title, label):
    plt.figure(figsize=(8, 4.5))
    sc = plt.scatter(x_space, y_space, c=np.asarray(values).reshape(-1), s=10)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(title)
    plt.axis("equal")
    plt.colorbar(sc, label=label)
    plt.tight_layout()
    plt.show()

plot_2d_field(u_snap, f"Reference u field | t={t_snap:.4g}", "u")
plot_2d_field(v_snap, f"Reference v field | t={t_snap:.4g}", "v")
plot_2d_field(p_snap, f"Reference pressure field | t={t_snap:.4g}", "p")
plot_2d_field(speed_snap, f"Reference velocity magnitude | t={t_snap:.4g}", "|velocity|")



这些图就是“完整 reference 空间场”。

注意：这里画的是数据集本身的参考解，不是模型预测。如果加载训练后的 `model.pt`，再对同一批 `(x,y,t)` 求值，就能得到 reconstructed/predicted field。


## 7. 速度矢量场：同时看方向和大小

In [ ]:

# 5000 个箭头会太密，所以只为了可视化抽稀。
QUIVER_TARGET = 500
stride = max(1, N // QUIVER_TARGET)
qidx = np.arange(0, N, stride)

plt.figure(figsize=(9, 4.8))
plt.quiver(
    x_space[qidx],
    y_space[qidx],
    u_snap[qidx],
    v_snap[qidx],
    speed_snap[qidx],
)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Reference velocity vectors | t={t_snap:.4g}")
plt.axis("equal")
plt.tight_layout()
plt.show()



## 8. 三维图：`x-y-场值`

下面的第三轴分别是 $u/v/p/|\mathbf{v}|$ 的数值，**不是物理空间的 z**。


In [ ]:

def plot_3d_field_cloud(values, title, zlabel, max_points=5000):
    values = np.asarray(values).reshape(-1)
    if N > max_points:
        idx = np.linspace(0, N - 1, max_points, dtype=int)
    else:
        idx = np.arange(N)

    fig = plt.figure(figsize=(8, 5.5))
    ax = fig.add_subplot(111, projection="3d")
    sc = ax.scatter(
        x_space[idx],
        y_space[idx],
        values[idx],
        c=values[idx],
        s=5,
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel(zlabel)
    ax.set_title(title)
    fig.colorbar(sc, ax=ax, shrink=0.7, label=zlabel)
    plt.tight_layout()
    plt.show()

plot_3d_field_cloud(u_snap, f"3D value cloud: u(x,y) | t={t_snap:.4g}", "u")
plot_3d_field_cloud(v_snap, f"3D value cloud: v(x,y) | t={t_snap:.4g}", "v")
plot_3d_field_cloud(p_snap, f"3D value cloud: p(x,y) | t={t_snap:.4g}", "p")
plot_3d_field_cloud(speed_snap, f"3D value cloud: |velocity|(x,y) | t={t_snap:.4g}", "|velocity|")



### 可选：用三角剖分画更像“曲面”的三维图

这是**可视化插值/连面**，不是数据集新增了更多 ground-truth 点。若几何中存在空洞或障碍物，三角剖分可能跨越空洞连接，因此科研定量分析仍以原始点值为准。


In [ ]:

tri = mtri.Triangulation(x_space, y_space)

fig = plt.figure(figsize=(8, 5.5))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_trisurf(
    tri,
    speed_snap,
    linewidth=0.0,
    antialiased=True,
)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("|velocity|")
ax.set_title(f"Triangulated velocity-magnitude surface | t={t_snap:.4g}")
plt.tight_layout()
plt.show()


## 9. 时间维度：固定一个空间点，看它的时序

In [ ]:

# 可以自行改 POINT_INDEX。
POINT_INDEX = min(N // 2, N - 1)

print(
    "POINT_INDEX:", POINT_INDEX,
    " coordinate=(", float(x_space[POINT_INDEX]), ",", float(y_space[POINT_INDEX]), ")"
)

plt.figure(figsize=(8, 4))
plt.plot(t_values, u_all[POINT_INDEX, :])
plt.xlabel("t")
plt.ylabel("u")
plt.title(f"u(t) at spatial point {POINT_INDEX}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(t_values, v_all[POINT_INDEX, :])
plt.xlabel("t")
plt.ylabel("v")
plt.title(f"v(t) at spatial point {POINT_INDEX}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(t_values, p_all[POINT_INDEX, :])
plt.xlabel("t")
plt.ylabel("p")
plt.title(f"p(t) at spatial point {POINT_INDEX}")
plt.tight_layout()
plt.show()



## 10. 当前实验的 800 个监督点到底多稀疏

当前 `load_training_data` 的核心逻辑是：

1. 把全部 $N\times T$ 个 `(x,y,t)` 展开；
2. 从中无放回随机抽 `N_TRAIN=800`；
3. 训练监督使用这些点的 `u/v`；
4. `p_star` 不进入 data loss，只在最终 evaluation 中使用。


In [ ]:

SEED = 0
N_TRAIN = 800

rng = np.random.RandomState(SEED)
train_idx = rng.choice(N * T, N_TRAIN, replace=False)

coverage = N_TRAIN / (N * T)
print("All space-time reference points:", N * T)
print("Supervised sampled points      :", N_TRAIN)
print(f"Direct supervised-point ratio  : {coverage:.6%}")

x_train = x_flat[train_idx]
y_train = y_flat[train_idx]
t_train = t_flat[train_idx]
u_train = u_flat[train_idx]
v_train = v_flat[train_idx]
p_reference_at_train = p_flat[train_idx]  # 仅用于理解；当前训练不把它作为 p label。

print()
print("Training batch conceptually uses: x, y, t, u, v")
print("p exists in the MAT reference but is NOT used by current data loss.")


In [ ]:

# 真正的 3D 输入域可视化：x-y-t。
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
speed_train = np.sqrt(u_train**2 + v_train**2)
sc = ax.scatter(
    x_train,
    y_train,
    t_train,
    c=speed_train,
    s=10,
)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("t")
ax.set_title(f"800 sparse supervised samples in space-time | seed={SEED}")
fig.colorbar(sc, ax=ax, shrink=0.7, label="reference |velocity|")
plt.tight_layout()
plt.show()


In [ ]:

# 每个时间快照实际上被抽到了多少个监督点？
# 通过最接近的时间索引统计。
time_index_of_sample = np.argmin(
    np.abs(t_train[:, None] - t_values[None, :]),
    axis=1,
)
counts = np.bincount(time_index_of_sample, minlength=T)

print("sample count per time snapshot:")
print("  min :", int(counts.min()))
print("  max :", int(counts.max()))
print("  mean:", float(counts.mean()))
print("  zero-observation snapshots:", int(np.sum(counts == 0)), "/", T)

plt.figure(figsize=(9, 4))
plt.plot(np.arange(T), counts)
plt.xlabel("snapshot index")
plt.ylabel("# sampled supervised points")
plt.title("How the 800 observations are distributed over time")
plt.tight_layout()
plt.show()



## 11. 时空全局可视化：`x-y-t` + 场值颜色

下面从若干时间快照、若干空间点中抽一部分，只是为了让 3D 图可读。

三个坐标轴：

$$
(x,y,t)
$$

才是当前问题真正的三维**输入域**；颜色表示 velocity magnitude。


In [ ]:

N_TIME_VIS = min(12, T)
N_SPACE_VIS = min(400, N)

time_vis_idx = np.linspace(0, T - 1, N_TIME_VIS, dtype=int)
space_vis_idx = np.linspace(0, N - 1, N_SPACE_VIS, dtype=int)

xx = []
yy = []
tt = []
cc = []

for k in time_vis_idx:
    xx.append(x_space[space_vis_idx])
    yy.append(y_space[space_vis_idx])
    tt.append(np.full(N_SPACE_VIS, t_values[k]))
    cc.append(speed_all[space_vis_idx, k])

xx = np.concatenate(xx)
yy = np.concatenate(yy)
tt = np.concatenate(tt)
cc = np.concatenate(cc)

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(xx, yy, tt, c=cc, s=4)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("t")
ax.set_title("Reference field in the x-y-t domain")
fig.colorbar(sc, ax=ax, shrink=0.7, label="|velocity|")
plt.tight_layout()
plt.show()



## 12. 可选动画：看圆柱尾流随时间变化

默认关闭，避免 notebook 一运行就生成大量 HTML。

把 `MAKE_ANIMATION=True` 后重新运行本 cell。为了控制体积，默认每隔若干 snapshot 取一帧。


In [ ]:

MAKE_ANIMATION = True

if MAKE_ANIMATION:
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML

    frame_stride = max(1, T // 40)
    frames = np.arange(0, T, frame_stride)

    fig = plt.figure(figsize=(8, 4.5))
    ax = fig.add_subplot(111)
    sc = ax.scatter(
        x_space,
        y_space,
        c=u_all[:, frames[0]],
        s=10,
        vmin=float(np.min(u_all)),
        vmax=float(np.max(u_all)),
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")
    cb = fig.colorbar(sc, ax=ax, label="u")

    def update(frame_idx):
        k = int(frames[frame_idx])
        sc.set_array(u_all[:, k])
        ax.set_title(f"Reference u field | snapshot={k}, t={t_values[k]:.4g}")
        return (sc,)

    animation = FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=150,
        blit=False,
    )
    plt.close(fig)
    display(HTML(animation.to_jshtml()))
else:
    print("Animation skipped. Set MAKE_ANIMATION=True to enable.")



# 13. 可选：加载训练后的 `model.pt`，真正看“模型恢复的整个场”

这一节把概念串起来：

```text
model.pt
   ↓
恢复网络参数
   ↓
输入完整 X_star + 一个指定时间 t
   ↓
得到 predicted u/v/p
   ↓
和 MAT 中的 reference u/v/p 比较
   ↓
Absolute Error
```

默认路径指向当前 ConFIG-4 seed=0 的 checkpoint。如果该文件不存在，本节会自动跳过，不影响前面的数据集理解。


In [ ]:

MODEL_PATH = Path(os.environ.get(
    "PINNSFORMER_MODEL_PT",
    "/home/simplexity/cyt/pinnsformer-main/pinnsformer-config/"
    "outputs/navier_stokes_4loss_multiseed/seed_0/config_4loss/model.pt"
))

COMMON_ROOT = Path(os.environ.get(
    "PINNSFORMER_CONFIG_ROOT",
    "/home/simplexity/cyt/pinnsformer-main/pinnsformer-config"
))

print("MODEL_PATH:", MODEL_PATH)
print("exists    :", MODEL_PATH.exists())
print("COMMON_ROOT:", COMMON_ROOT)


In [ ]:

MODEL_PREDICTION = None

if MODEL_PATH.exists() and (COMMON_ROOT / "navier_stokes_common.py").exists():
    import torch

    sys.path.insert(0, str(COMMON_ROOT))
    from navier_stokes_common import PINNsformer, make_time_sequence

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print("Inference device:", device)

    model = PINNsformer(
        d_out=2,
        d_hidden=512,
        d_model=32,
        N=1,
        heads=2,
    ).to(device)

    # state = torch.load(MODEL_PATH, map_location=device)
    state = torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True,
    )
    model.load_state_dict(state)
    model.eval()

    NUM_STEP = 5
    TIME_STEP = 1e-2

    x_np = np.expand_dims(np.tile(X_star[:, 0:1], NUM_STEP), -1)
    y_np = np.expand_dims(np.tile(X_star[:, 1:2], NUM_STEP), -1)
    t_base = np.full((N, 1), t_values[SNAPSHOT], dtype=np.float64)
    t_np = make_time_sequence(t_base, num_step=NUM_STEP, step=TIME_STEP)

    x_t = torch.tensor(x_np, dtype=torch.float32, requires_grad=True, device=device)
    y_t = torch.tensor(y_np, dtype=torch.float32, requires_grad=True, device=device)
    t_t = torch.tensor(t_np, dtype=torch.float32, requires_grad=True, device=device)

    psi_p = model(x_t, y_t, t_t)
    psi = psi_p[:, :, 0:1]
    p_pred_seq = psi_p[:, :, 1:2]

    u_pred_seq = torch.autograd.grad(
        psi,
        y_t,
        grad_outputs=torch.ones_like(psi),
        retain_graph=True,
        create_graph=False,
    )[0]

    v_pred_seq = -torch.autograd.grad(
        psi,
        x_t,
        grad_outputs=torch.ones_like(psi),
        retain_graph=False,
        create_graph=False,
    )[0]

    MODEL_PREDICTION = {
        "u": u_pred_seq.detach().cpu().numpy()[:, 0].reshape(-1),
        "v": v_pred_seq.detach().cpu().numpy()[:, 0].reshape(-1),
        "p": p_pred_seq.detach().cpu().numpy()[:, 0].reshape(-1),
    }

    print("Model prediction generated for", N, "spatial coordinates.")
else:
    print(
        "Checkpoint/common module not found -> skip model reconstruction. "
        "Set PINNSFORMER_MODEL_PT and PINNSFORMER_CONFIG_ROOT if needed."
    )


In [ ]:

def relative_l2(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return float(
        np.linalg.norm(y_true - y_pred)
        / np.linalg.norm(y_true)
    )

if MODEL_PREDICTION is not None:
    truth_snapshot = {
        "u": u_snap,
        "v": v_snap,
        "p": p_snap,
    }

    p_pred = MODEL_PREDICTION["p"]
    p_pred_aligned = (
        p_pred - np.mean(p_pred) + np.mean(p_snap)
    )

    print("Snapshot field Relative L2:")
    print("u:", relative_l2(u_snap, MODEL_PREDICTION["u"]))
    print("v:", relative_l2(v_snap, MODEL_PREDICTION["v"]))
    print("p raw:", relative_l2(p_snap, p_pred))
    print("p mean-aligned:", relative_l2(p_snap, p_pred_aligned))


In [ ]:
if MODEL_PREDICTION is not None:
    # 模型恢复的 u 场
    plot_2d_field(
        MODEL_PREDICTION["u"],
        f"PINNsFormer reconstructed u field | t={t_snap:.4g}",
        "u_pred",
    )

    # 模型恢复的 v 场
    plot_2d_field(
        MODEL_PREDICTION["v"],
        f"PINNsFormer reconstructed v field | t={t_snap:.4g}",
        "v_pred",
    )

    # 模型恢复的 pressure 场
    plot_2d_field(
        MODEL_PREDICTION["p"],
        f"PINNsFormer reconstructed pressure field | t={t_snap:.4g}",
        "p_pred",
    )

    # 恢复的速度模长
    speed_pred = np.sqrt(
        MODEL_PREDICTION["u"]**2
        + MODEL_PREDICTION["v"]**2
    )

    plot_2d_field(
        speed_pred,
        f"PINNsFormer reconstructed velocity magnitude | t={t_snap:.4g}",
        "|velocity| pred",
    )

## 14. Reference / Prediction / Absolute Error 逐个看

In [ ]:

def plot_reference_prediction_error(field, truth, pred):
    plot_2d_field(
        truth,
        f"{field} reference | t={t_snap:.4g}",
        field,
    )
    plot_2d_field(
        pred,
        f"{field} model prediction | t={t_snap:.4g}",
        field,
    )
    plot_2d_field(
        np.abs(np.asarray(truth).reshape(-1) - np.asarray(pred).reshape(-1)),
        f"{field} absolute error | t={t_snap:.4g}",
        "absolute error",
    )

if MODEL_PREDICTION is not None:
    plot_reference_prediction_error(
        "u",
        u_snap,
        MODEL_PREDICTION["u"],
    )
    plot_reference_prediction_error(
        "v",
        v_snap,
        MODEL_PREDICTION["v"],
    )
    plot_reference_prediction_error(
        "p (raw)",
        p_snap,
        MODEL_PREDICTION["p"],
    )

    p_aligned = (
        MODEL_PREDICTION["p"]
        - np.mean(MODEL_PREDICTION["p"])
        + np.mean(p_snap)
    )
    plot_reference_prediction_error(
        "p (mean-aligned)",
        p_snap,
        p_aligned,
    )
else:
    print("No loaded model -> skip prediction/error figures.")



## 15. 如何读 Absolute Error 图

对任一场变量 $q$：

$$
E_q(x,y)
=
|q_{\mathrm{reference}}(x,y)-q_{\mathrm{prediction}}(x,y)|.
$$

因此：

$$
E_q\rightarrow0
$$

越好。

**不要只凭“颜色浅/深”判断好坏，要看 colorbar 数值。** 本 notebook 没有手动固定某一种 colormap；原则永远是 error 数值越小越好。

另外：

- `reference field` = `.mat` 中的完整参考场；
- `prediction field` = `model.pt` 对完整坐标求值得到的恢复场；
- `absolute error` = 上面两者逐点相减后的绝对值。

所以 `model.pt` 本身不是一张场图，而是一个可继续求值的神经函数参数集合。



# 16. 看完这个 notebook 后应该能回答的问题

1. 这是 3D 物理空间吗？  
   **不是。是二维 $(x,y)$ + 时间 $t$。**

2. `.mat` 里有没有完整的 $u/v/p$？  
   **有。它们作为 benchmark reference / ground truth 存在。**

3. 当前训练把完整 $u/v/p$ 都喂给网络了吗？  
   **没有。当前随机抽 800 个时空位置，只直接监督 $u/v$，pressure 不直接监督。**

4. PINNsFormer 学到的是什么？  
   **一个神经网络表示的连续可微时空函数 $F_\theta(x,y,t)$。**

5. `.pt` 是什么？  
   **这个函数的网络参数，不是固定的一张预测图。**

6. “恢复整个场”是什么意思？  
   **把训练好的函数在完整空间坐标、指定时间上求值，得到全场 $u/v/p$。**

7. Absolute Error 是什么？  
   **恢复场和 reference field 的逐点绝对误差。数值越接近 0 越好。**

8. 当前是纯自监督吗？  
   **不是。是稀疏 $u/v$ 监督 + 无标签 PDE residual 约束的 physics-informed learning。**
